# 03 · Problem 1: cross-generator detection baselines
**Runtime: GPU (T4).** Runtime → Change runtime type → *T4 GPU*.
**Input:** `manifests/processed_index.csv`, `manifests/splits.csv`, `processed/` (from notebook 02).

**Task:** train real-vs-fake on **Stable Diffusion v1.4 only**, then test on all 8 GenImage
generators (7 never seen in training). Main metric: **AUROC per generator**.

| Role | Model | What is trained |
|---|---|---|
| Challenger A1 | frozen **CLIP ViT-L/14** features + logistic regression | only the linear classifier |
| Challenger A2 | frozen **DINOv2 ViT-B/14** features + logistic regression | only the linear classifier |
| Champion B | ImageNet-pretrained **ResNet-50**, fully fine-tuned | all weights |

Sources: ResNet (He et al. 2016, arXiv 1512.03385; torchvision weights), CLIP (Radford et al. 2021,
arXiv 2103.00020; HF `openai/clip-vit-large-patch14`), DINOv2 (Oquab et al. 2023, arXiv 2304.07193;
HF `facebook/dinov2-base`). The frozen-CLIP linear probe follows UnivFD (Ojha et al. 2023,
arXiv 2302.10174); crop-without-resize training follows Wang et al. 2020 (arXiv 1912.11035).

In [ ]:
# ===== Standard setup cell (identical in every notebook) =====
# What it does: (1) in Colab, clone the repo the first time and `git pull` afterwards,
# (2) make src/ importable, (3) run src.colab_setup.setup(): mount Drive, install deps
# (incl. apt p7zip-full), set seeds, print GPU info, create the Drive output folders.
%load_ext autoreload
%autoreload 2
import os, sys, subprocess

REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/genimage-detection.git"  # <-- edit once after creating the repo
REPO_DIR = "/content/genimage-detection"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
        print("[setup] git pull:", (r.stdout or r.stderr).strip())
else:  # local run: the notebook lives in <repo>/notebooks
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.colab_setup import setup
cfg, P = setup()   # cfg = dict from configs/config.yaml, P = resolved paths

## 1 · GPU check

In [ ]:
import torch

ALLOW_CPU = False   # True only for debugging: CPU would take many hours
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU OK:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    msg = ("NO GPU FOUND. In Colab: Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart "
           "and run all again.")
    if not ALLOW_CPU:
        raise RuntimeError(msg)
    print("WARNING:", msg)

In [ ]:
import gc

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from torch.utils.data import DataLoader

from src import datasets as ds
from src import eda_utils as eu
from src import evaluate as ev
from src import models as mdl
from src import train as tr
from src.config import generator_names, get_logger

eu.set_style()
pd.set_option("display.width", 200, "display.max_columns", 30)
log = get_logger("03_problem1_baselines", P.logs)
SEED = cfg["project"]["seed"]
NAMES = generator_names(cfg)
GEN_ORDER = [g["folder"] for g in cfg["generators"]]
TRAIN_GEN = cfg["train_generator"]
THR = cfg["evaluation"]["threshold"]
FEAT = cfg["features"]
RCFG = cfg["training"]["resnet50"]


def save(fig, name):
    eu.savefig(fig, f"03_{name}", P.figures)
    plt.show()


# Fixed row order (sorted by sample_id) so cached features always line up with this table.
index = ds.load_index(P.manifests).sort_values("sample_id").reset_index(drop=True)
display(index.groupby(["split_set", "generator_name", "label"]).size().unstack("label"))

### Copy the processed images to the local disk
~18k small JPEGs. Reading them from Drive in every epoch is slow, so we copy them once to the VM's
local disk (`/content/work/processed`, a few hundred MB). Re-runs only copy missing files.

In [ ]:
if cfg["training"]["copy_processed_to_local"]:
    n = ds.stage_locally(index["processed_relpath"], P.project_root, P.work_dir)
    DATA_ROOT = P.work_dir
    print(f"Copied {n} new files; reading images from {DATA_ROOT}")
else:
    DATA_ROOT = P.project_root
masks = {s: (index["split_set"] == s).to_numpy() for s in ["train", "val", "test"]}
y_all = index["y"].to_numpy()

---
## 2 · Challengers: frozen CLIP / DINOv2 features + logistic regression
1. Embed every image once with the frozen backbone. Embeddings are cached on Drive in
   `features/<model>/` as `.npy` shards, so this runs only once (and resumes after a disconnect).
2. Fit a logistic regression on the SD1.4 **train** split for several regularisation strengths C;
   keep the C with the best AUROC on the SD1.4 **val** split.
3. Score the test sets of all 8 generators.

In [ ]:
probe_preds, probe_tuning = [], []
for kind, model_name in [("clip", "CLIP ViT-L/14 + LR"), ("dinov2", "DINOv2 ViT-B/14 + LR")]:
    log.info(f"=== {model_name} ===")
    dset = ds.GenImageDataset(index, DATA_ROOT, ds.feature_transforms(kind, cfg["preprocess"]["image_size"]))
    extractor = mdl.build_feature_extractor(kind, cfg)
    X = mdl.extract_features_cached(extractor, dset, P.features / kind, batch_size=FEAT["batch_size"],
                                    shard_size=FEAT["shard_size"], num_workers=FEAT["num_workers"], device=DEVICE)
    del extractor
    gc.collect()
    torch.cuda.empty_cache() if DEVICE == "cuda" else None
    print(f"{kind} features: {X.shape}")

    clf, tuning = tr.train_linear_probe(X[masks["train"]], y_all[masks["train"]], X[masks["val"]], y_all[masks["val"]],
                                        cfg["training"]["linear_probe"]["C_grid"],
                                        cfg["training"]["linear_probe"]["max_iter"], SEED, logger=log)
    tuning.insert(0, "model", model_name)
    probe_tuning.append(tuning)
    joblib.dump(clf, P.checkpoints / f"linear_probe_{kind}.joblib")
    best_C = tuning.loc[tuning["val_auroc"].idxmax(), "C"]
    print(f"Best C = {best_C}; saved {P.checkpoints / f'linear_probe_{kind}.joblib'}")

    ev_mask = masks["val"] | masks["test"]
    probe_preds.append(index.loc[ev_mask, ["sample_id", "generator", "split_set", "y"]]
                       .assign(model=model_name, score=clf.predict_proba(X[ev_mask])[:, 1]))

tuning_df = pd.concat(probe_tuning, ignore_index=True)
ev.save_table(tuning_df, P.results / "problem1_linear_probe_tuning.csv")
fig, ax = plt.subplots(figsize=(6, 3.5))
for m, g in tuning_df.groupby("model"):
    ax.plot(g["C"], g["val_auroc"], marker="o", label=m)
ax.set(xscale="log", xlabel="C (inverse regularisation)", ylabel="SD1.4 val AUROC", title="Linear probe: tuning C")
ax.legend(fontsize=8)
save(fig, "linear_probe_tuning")

---
## 3 · Champion: fine-tuned ResNet-50
- ImageNet-pretrained (torchvision `IMAGENET1K_V2`), last layer replaced by one logit (fake score).
- Loss: binary cross-entropy; AdamW + cosine learning-rate schedule; mixed precision (AMP).
- Augmentation: **random crop (no resizing) + horizontal flip only**. No JPEG/blur augmentation:
  that's Problem 2. Evaluation uses a center crop of the same size.
- Early stopping on SD1.4 val AUROC. `last.pt` is saved every epoch: if Colab disconnects, re-run
  this cell and training **resumes**. `best.pt` = best val AUROC.

In [ ]:
g = torch.Generator().manual_seed(SEED)
train_ds = ds.GenImageDataset(index.loc[masks["train"]], DATA_ROOT, ds.resnet_transforms(RCFG["crop_size"], train=True))
val_ds = ds.GenImageDataset(index.loc[masks["val"]], DATA_ROOT, ds.resnet_transforms(RCFG["crop_size"], train=False))
test_ds = ds.GenImageDataset(index.loc[masks["test"]], DATA_ROOT, ds.resnet_transforms(RCFG["crop_size"], train=False))
dl_kw = dict(batch_size=RCFG["batch_size"], num_workers=RCFG["num_workers"], pin_memory=DEVICE == "cuda")
train_dl = DataLoader(train_ds, shuffle=True, drop_last=True, generator=g, **dl_kw)
val_dl = DataLoader(val_ds, shuffle=False, **dl_kw)
test_dl = DataLoader(test_ds, shuffle=False, **dl_kw)
print(f"train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)} images")

CKPT = P.checkpoints / "resnet50"
resnet = mdl.build_resnet50(RCFG["weights"])
history = tr.train_resnet(resnet, train_dl, val_dl, val_ds.df["y"].to_numpy(), CKPT, epochs=RCFG["epochs"],
                          lr=RCFG["lr"], weight_decay=RCFG["weight_decay"], patience=RCFG["early_stopping_patience"],
                          amp=RCFG["amp"], device=DEVICE, logger=log)
display(history)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(history["epoch"], history["train_loss"], marker="o", label="train")
axes[0].plot(history["epoch"], history["val_loss"], marker="o", label="val")
axes[0].set(title="ResNet-50 loss", xlabel="epoch", ylabel="BCE")
axes[0].legend()
axes[1].plot(history["epoch"], history["val_auroc"], marker="o", label="val AUROC")
axes[1].plot(history["epoch"], history["val_acc"], marker="o", label="val accuracy")
axes[1].set(title="ResNet-50 SD1.4 validation", xlabel="epoch", ylim=(0.4, 1.01))
axes[1].legend()
save(fig, "resnet50_training_curves")

In [ ]:
# Evaluate the BEST checkpoint (not the last epoch).
best = torch.load(CKPT / "best.pt", map_location=DEVICE, weights_only=False)
resnet.load_state_dict(best["model"])
resnet.to(DEVICE)
print(f"Loaded best.pt from epoch {best['epoch'] + 1} (val AUROC {best['val_auroc']:.4f})")
resnet_preds = pd.concat([
    d.df[["sample_id", "generator", "split_set", "y"]].assign(
        model="ResNet-50 (fine-tuned)", score=tr.predict_scores(resnet, dl, DEVICE, RCFG["amp"]))
    for d, dl in [(val_ds, val_dl), (test_ds, test_dl)]], ignore_index=True)

---
## 4 · Evaluation per test generator
For each model and generator: accuracy, F1, **AUROC**, average precision, and the accuracy on real
and on fake images separately (threshold `evaluation.threshold` = 0.5 on P(fake)).
`mean_unseen` = average over the 7 generators not used in training.

Real-image accuracy shows false alarms; fake-image accuracy shows missed fakes. On unseen generators,
a detector that is "confused" usually keeps calling real images real but misses the new fakes.

In [ ]:
MODEL_ORDER = ["ResNet-50 (fine-tuned)", "CLIP ViT-L/14 + LR", "DINOv2 ViT-B/14 + LR"]
preds = pd.concat([resnet_preds] + probe_preds, ignore_index=True)
preds.to_csv(P.results / "problem1_predictions.csv", index=False)

val_res = ev.per_generator_table(preds.loc[preds["split_set"] == "val"], THR)
print("SD1.4 validation split (in-distribution, used for model selection):")
display(val_res[["model", "accuracy", "f1", "auroc", "real_acc", "fake_acc"]].round(4))

test_preds = preds.loc[preds["split_set"] == "test"]
results = ev.per_generator_table(test_preds, THR, GEN_ORDER, train_generator=TRAIN_GEN)
results.insert(2, "generator_name", results["generator"].map(NAMES).fillna(results["generator"]))
results["seen_in_training"] = results["generator"] == TRAIN_GEN
ev.save_table(results, P.results / "problem1_results.csv")

for metric in ["auroc", "accuracy", "f1", "real_acc", "fake_acc"]:
    piv = results.pivot(index="model", columns="generator_name", values=metric)
    piv = piv[[NAMES[g] for g in GEN_ORDER if NAMES[g] in piv.columns] + ["mean_unseen"]].reindex(MODEL_ORDER)
    print(f"\n{metric.upper()} (rows = models, columns = test generators; {NAMES[TRAIN_GEN]} was seen in training)")
    display(piv.style.format("{:.3f}").background_gradient(cmap="RdYlGn", vmin=0.5, vmax=1.0, axis=None))
    if metric == "auroc":
        piv.to_csv(P.results / "problem1_auroc_table.csv")

In [ ]:
fig = ev.plot_metric_bars(results.loc[results["generator"] != "mean_unseen"], "auroc", GEN_ORDER, NAMES)
save(fig, "auroc_per_generator")
fig = ev.plot_metric_bars(results.loc[results["generator"] != "mean_unseen"], "accuracy", GEN_ORDER, NAMES)
save(fig, "accuracy_per_generator")

In [ ]:
fig = ev.plot_roc_per_generator(test_preds, GEN_ORDER, names=NAMES)
save(fig, "roc_per_generator")

In [ ]:
for m in MODEL_ORDER:
    fig = ev.plot_confusion_grid(test_preds, m, GEN_ORDER, THR, names=NAMES)
    save(fig, f"confusion_{m.split()[0].lower().replace('-', '')}")

In [ ]:
# Real vs fake accuracy per generator: where do the errors come from?
r = results.loc[results["generator"] != "mean_unseen"].copy()
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(5 * len(MODEL_ORDER), 3.6), sharey=True)
for ax, m in zip(axes, MODEL_ORDER):
    g = r.loc[r["model"] == m].set_index("generator_name")[["real_acc", "fake_acc"]]
    g.reindex([NAMES[x] for x in GEN_ORDER]).plot.bar(ax=ax, color=[eu.REAL_COLOR, eu.FAKE_COLOR])
    ax.set(title=m, xlabel="", ylim=(0, 1.02))
    ax.tick_params(axis="x", rotation=45)
save(fig, "real_vs_fake_accuracy")

---
## 5 · Model comparison (edit after running)
**Algorithms and their trade-offs on this data**

| | Fine-tuned ResNet-50 (champion) | Frozen CLIP / DINOv2 + linear probe (challengers) |
|---|---|---|
| What is learned | all ~25M weights adapt to SD1.4 artifacts | only ~770 weights (one linear layer) on fixed features |
| Pros | learns low-level, pixel-level traces; strong in-distribution; cheap inference | features come from huge, diverse pre-training, not from SD1.4 → less over-fitting to one generator; training is seconds once features are cached |
| Cons | can over-fit to one generator's fingerprint → may drop on unseen generators; needs a GPU to train | features are semantic and resolution-limited (224 px, patch 14) and may miss subtle high-frequency traces; heavy backbones (ViT-L) at inference |

**Results** (fill in from the tables above):
- In-distribution (SD1.4) AUROC: ResNet-50 … / CLIP … / DINOv2 …
- Mean AUROC on the 7 unseen generators: ResNet-50 … / CLIP … / DINOv2 …
- Generators where all models struggle: …  (hypothesis: e.g. BigGAN is a GAN, the others are diffusion
  models; Midjourney was downscaled 4.6× which removes high-frequency traces; …)
- SD1.5 and Wukong share SD1.4's latent-diffusion architecture → expected to be easy for every model.
- Which model generalises better and why: …
- Error type on unseen generators (real_acc vs fake_acc): …

---
## Findings (fill in after running)
- Training: epochs run, best epoch, val AUROC, training time per epoch:
- Linear probes: best C for CLIP / DINOv2, val AUROC:
- Per-generator AUROC highlights:
- Champion vs challengers on unseen generators:
- Surprises / failure cases (look at misclassified images):
- Next steps (see docs/NEXT_STEPS.md): Grad-CAM, robustness (Problem 2), raw-vs-processed training.